# 课后练习解答（02.02_environment_setup）

本解答对应章节课后练习，共 15 题。

### 问题1（单选题）

**题目：** 在 NPU 推理环境中，以下哪段导入与初始化顺序是正确的？
A. 先 import torch_npu，再 import torch
B. 先 import torch，再 import torch_npu，并执行一次 torch.npu.set_device(0)
C. 只 import torch_npu，无需 import torch
D. 先 import torch.cuda，再 import torch_npu

**解答：** B

**解析：** torch_npu 是对 torch 的插件式扩展，必须先完成 torch 导入；显式 set_device 可避免多卡时的默认设备歧义。


### 问题2（单选题）

**题目：** npu-smi 显示 8 张 NPU 正常，但 torch.npu.is_available() 返回 False，最可能的原因是？
A. torch/CANN/torch_npu 版本不匹配或驱动未正确加载
B. 数据集路径错误
C. 模型未定义
D. 学习率过大

**解答：** A

**解析：** is_available() 依赖 torch_npu 与 CANN runtime 能否成功初始化，与数据集、模型、超参数无关。


### 问题3（多选题）

**题目：** 以下哪些检查组合能较完整地验证 CANN 与 torch_npu 环境？
A. npu-smi info 查看设备状态
B. torch.npu.is_available() 与 torch.npu.device_count()
C. 打印 torch.__version__ 与 torch_npu.__version__
D. 创建 torch.ones(4, device="npu:0") 并回拷到 CPU

**解答：** ABCD

**解析：** 设备可见性、版本兼容、实际算子往返都通过，才能说明环境可运行。


### 问题4（多选题）

**题目：** Docker 容器共享内存过小可能引发哪些现象？
A. DataLoader 多进程 worker 报共享内存相关 OSError
B. /dev/shm 写入失败
C. 模型权重文件损坏
D. 数据加载速度异常下降

**解答：** ABD

**解析：** 共享内存不足影响 worker 间数据传输与临时文件，不会直接损坏已落盘的权重文件。


### 问题5（判断题）

**题目：** 先导入 torch 再导入 torch_npu，是 torch_npu 正确使用的必要条件之一。

**解答：** 对

**解析：** torch_npu 需要 patch torch 的设备扩展接口，因此 torch 必须先加载。


### 问题6（判断题）

**题目：** 在只有 NPU 的机器上，torch.cuda.is_available() 返回 True，因此可以直接沿用 CUDA 设备选择逻辑。

**解答：** 错

**解析：** 纯 NPU 环境通常没有 CUDA runtime，该调用返回 False，设备选择必须优先使用 torch.npu.is_available()。


### 问题7（填空题）

**题目：** 在 CANNLab 环境中，查看 NPU 设备数量、健康状态与显存占用应使用命令 ____。

**解答：** npu-smi info


### 问题8（填空题）

**题目：** 限制当前进程可见 NPU 卡的环境变量是 ____。

**解答：** ASCEND_RT_VISIBLE_DEVICES


### 问题9（简答题）

**题目：** 为什么推荐使用 conda 为 CANN/PyTorch 实验创建独立环境？请从依赖隔离、版本锁定和复现三个角度说明。

**解答：** conda 可将 Python 解释器与包集合隔离到独立环境，避免不同课程实验互相污染；通过 environment.yml 或 requirements 可锁定 torch、torch_npu、CANN 配套版本；换机器后按相同配置重建环境，能显著提高结果可复现性。


### 问题10（简答题）

**题目：** 安装或升级 CANN、驱动或 torch_npu 后，为什么必须重启 Jupyter kernel 才能生效？

**解答：** Python 进程启动时才会加载并初始化动态库与环境变量；不重启 kernel，进程内仍持有旧库句柄和旧配置，升级不会在当前会话生效。


### 问题11（代码设计题）

**题目：** 编写 get_device()，要求优先返回 npu:0，不可用时回退 cpu，并打印 torch、torch_npu 版本与 NPU 数量。

**解答：** ```python
import torch
import torch_npu

def get_device():
    print("torch", torch.__version__)
    print("torch_npu", torch_npu.__version__)
    if torch.npu.is_available():
        print("npu_count", torch.npu.device_count())
        return "npu:0"
    return "cpu"
```


### 问题12（单选题）

**题目：** 在 NPU 上执行 torch.npu.synchronize() 的主要目的是？
A. 释放显存
B. 等待异步 kernel 全部完成，保证后续计时与读取结果可靠
C. 触发图编译
D. 清空缓存

**解答：** B

**解析：** NPU 算子通常异步下发，同步点是确保执行完成、结果可见的边界。


### 问题13（多选题）

**题目：** 日志出现 Permission mismatch: The owner of ... does not match 时，常见原因与处理包括？
A. 包由 root 安装而当前用户非 root
B. 文件 owner 与运行用户不一致
C. 通过 chown 或重建虚拟环境缓解
D. 直接删除该文件即可彻底解决

**解答：** ABC

**解析：** 权限不匹配通常源于安装用户与运行用户不同；删除库文件会破坏环境，不是解决方案。


### 问题14（判断题）

**题目：** CANNLab 镜像预装了 torch/torch_npu，因此实验中不需要再安装任何其他 Python 依赖。

**解答：** 错

**解析：** 镜像只保证核心环境，matplotlib、datasets、trl、peft 等上层依赖仍可能缺失或需要指定版本。


### 问题15（简答题）

**题目：** DataLoader 设置 num_workers=8 后容器报共享内存不足，请给出至少三种排查或解决思路。

**解答：** 1) 调小 num_workers 或改用 persistent_workers=False；2) 启动容器时增大 /dev/shm，如 --shm-size=16g；3) 使用 pin_memory=False 或改用文件系统缓存；4) 检查是否在 DataLoader worker 中重复加载大对象。
